# ¿Cómo sabes que tu RAG funciona?
## Métricas de evaluación desde cero

Juan Manuel Carballo · github.com/jmanuelc87

---

## Construir un RAG toma una tarde; saber si responde bien, no

Retrieval Augmented Generation (RAG) es un patrón que conecta un LLM con una fuente de datos externa: un retriever busca contexto relevante y el LLM genera la respuesta con eso.

<center>
<img width="700" src="../assets/img/RAG.png">
</center>

- Tienes retriever + vector store + LLM. Responde.
- ¿Responde **con lo que recuperó** o con lo que ya sabía?
- ¿Cómo lo mides en 500 preguntas sin leerlas todas?

---

## Un LLM-as-a-Judge es un modelo que califica a otro… con reglas explícitas

Un LLM-as-a-Judge no es "pregúntale a un modelo si está bien": es un patrón con piezas concretas.

- **Entradas:** salida evaluada + contexto recuperado + rúbrica de evaluación
- **Salida:** JSON parseable (score + justificación) — apto para ser leído por una máquina
- Después del juez: revisión humana y una **capa de calibración**
- Se calibra con técnicas de prompt y, cuando hay acceso al modelo, con clasificadores (ver más adelante)

<center>
<img width="600" src="../assets/img/Evaluacion.png">
</center>

---

## Tres formas de pedirle un veredicto al juez

Faithfulness usa veredictos binarios por afirmación, no una escala 1–5.

| Tipo | Qué recibe el juez | Qué devuelve | Uso típico |
|---|---|---|---|
| Pairwise | dos respuestas | cuál es mejor | A/B de modelos |
| Pointwise | pregunta + respuesta | escala (1–5) | calidad general |
| **Binario (sí/no)** | una afirmación | sí / no | **faithfulness** |

---

## Faithfulness: ¿la respuesta está sostenida por el contexto recuperado?

- Faithfulness ∈ [0, 1]
- 1.0 → toda la respuesta se deriva del contexto
- Penaliza lo que la respuesta agrega o desvía del contexto recuperado
- **No** mide si la respuesta es correcta: mide si es fiel a lo recuperado
- La definición es la misma en RAGAS y DeepEval; **el cálculo no**

---

## RAGAS: cada afirmación debe estar *soportada* por el contexto

```
INPUT:
  q   : question           (string)
  a   : answer/output      (string)
  c   : context = passages (list[string])
  LLM : judge model
OUTPUT:
  F   : faithfulness score in [0, 1]

FUNCTION ragas_faithfulness(q, a, c, LLM):

  # Step 1 — Statement extraction FROM THE ANSWER
  S = LLM.extract_statements(q, a)

  # Step 2 — Verify each statement AGAINST THE CONTEXT
  supported = 0
  FOR s_i IN S:
      # "can s_i be inferred from context?" -> Yes/No (+reason)
      verdict = LLM.verify(s_i, c)
      # <-- POSITIVE support required
      IF verdict == YES:                
          supported += 1

  # Step 3 — Score
  F = supported / len(S)
  RETURN F
```

`F = #soportadas / #afirmaciones` — si el contexto no menciona la afirmación, cuenta como **NO soportada**.

---

## DeepEval: una afirmación solo falla si el contexto la *contradice*

```
INPUT:
  input             : user query (string)     # required param, NOT scored
  actual_output     : generated output (string)
  retrieval_context : passages (list[string])
  LLM               : judge model
OUTPUT:
  score  : faithfulness score in [0, 1]

FUNCTION deepeval_faithfulness(actual_output, retrieval_context, LLM):

  # Step 1 — Extract TRUTHS from the CONTEXT
  truths = LLM.generate_truths(retrieval_context, limit)

  # Step 2 — Extract CLAIMS from the OUTPUT
  claims = LLM.generate_claims(actual_output)

  # Step 3 — Per-claim verdict: does it CONTRADICT the truths?
  verdicts = []                          # each verdict in {"yes", "no", "idk"}
  FOR claim IN claims:
      v = LLM.generate_verdict(claim, truths)
          # "no"  ONLY if truths DIRECTLY contradict the claim
          # "idk" if not mentioned / unverifiable
          # "yes" if it agrees
      verdicts.append(v)

  # Step 4 — Score = fraction of claims NOT contradicted
  faithful = COUNT(v IN verdicts WHERE v != "no")   # <-- "yes" AND "idk" both pass
  score = faithful / len(verdicts)

  RETURN score
```

`yes` e `idk` aprueban; solo `no` penaliza.

---

## Misma métrica, dos preguntas distintas al juez

| | RAGAS | DeepEval |
|---|---|---|
| Unidad evaluada | statements de la respuesta | claims de la respuesta vs truths del contexto |
| Pregunta al juez | ¿se infiere del contexto? | ¿el contexto lo contradice? |
| Veredictos | sí / no | yes / no / idk |
| **Afirmación no mencionada** | **penaliza** | **aprueba** |
| Sesgo esperado | scores más bajos, más estricto | scores más altos, más permisivo |

---

## Implementación propia: faithfulness estilo RAGAS en ~15 líneas

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from syntok import segmenter


def split_sentences(text: str) -> list[str]:
    """Segment a text blob into sentences with syntok."""
    sentences: list[str] = []
    for paragraph in segmenter.analyze(text):
        for sentence in paragraph:
            rebuilt = "".join(token.spacing + token.value for token in sentence).strip()
            if rebuilt:
                sentences.append(rebuilt)
    return sentences

In [3]:
from pydantic import BaseModel


class RagasEntailment(BaseModel):
    """One claim's entailment verdict plus the judge's self-reported confidence."""

    entailed: bool = False
    justification: str = ""

In [4]:
DEFAULT_SYSTEM_PROMPT = (
    "Eres un evaluador experto de conversaciones entre usuarios y asistentes de "
    "IA. Evalúas un único turno según una rúbrica y devuelves una puntuación "
    "numérica junto con una justificación breve. Sé objetivo y responde siempre "
    "en español."
)

In [5]:
VERIFY_RAGAS = """\
¿Puede inferirse la siguiente afirmación a partir del contexto recuperado?
Devuelve:
- entailed: true si la afirmación se deduce del contexto, false si no se deduce o lo \
contradice.
- justification: una justificación breve en español.
Afirmación: {claim}"""

In [6]:
from openai import OpenAI

client = OpenAI()


def call_openai(messages: list[dict], schema: type[BaseModel]) -> BaseModel:
    """Wrapper alrededor de Structured Outputs: devuelve el modelo pydantic ya validado."""
    completion = client.chat.completions.parse(
        model="gpt-4o-mini",
        messages=messages,  # type: ignore[arg-type]
        response_format=schema,
    )
    parsed = completion.choices[0].message.parsed
    if parsed is None:
        raise ValueError("El modelo no devolvió una salida estructurada válida.")
    return parsed

In [7]:
# simplificación: split_sentences en vez de extracción de statements con LLM
# (se pierde: afirmaciones compuestas dentro de una misma oración)
def faithfulness_ragas(prompt: str, response: str, context: str):
    claims = split_sentences(response)

    supported = 0
    for claim in claims:
        content = VERIFY_RAGAS.format(**{"claim": claim})

        result = call_openai(
            messages=[
                {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": """Pregunta: {prompt}\n\nContexto: {context}\n\nContenido: {content}""".format(
                        **{"prompt": prompt, "context": context, "content": content}
                    ),
                },
            ],
            schema=RagasEntailment,
        )

        print(result)

        supported += int(result.entailed)  # type: ignore

    return supported / len(claims)

In [8]:
faithfulness_ragas(
    "Quien fue Nikola Tesla? y Cuáles fueron sus contribuciones?",
    "Fue un físico del siglo XIX y contribuyo al diseño de la corriente alterna",
    "Nikola Tesla fue un ingeniero, futurista e inventor serbio-estadounidense. Es conocido por sus contribuciones al diseño del sistema moderno de suministro eléctrico de corriente alterna. "
    "Nacido y criado en el Imperio austrohúngaro, Tesla estudió ingeniería y física en la década de 1870, aunque no obtuvo ningún título.",
)

entailed=True justification='El contexto menciona que Nikola Tesla fue un ingeniero y es conocido por sus contribuciones al sistema de corriente alterna, lo que respalda la afirmación de que contribuyó a su diseño.'


1.0

In [9]:
faithfulness_ragas(
    "Que dia es hoy?",
    "Hoy es Lunes 13 de Agosto de 2026",
    "La fecha de hoy es 11 de Agosto de 2026",
)

entailed=False justification='La afirmación indica que hoy es Lunes 13 de Agosto de 2026, pero el contexto señala que la fecha actual es 11 de Agosto de 2026. Por lo tanto, la afirmación no se deduce del contexto y se contradice.'


0.0

---

## Resultados RAGAS: soportado → 1.0, contradicho → 0.0

| Pregunta | Respuesta | Contexto (resumen) | Veredicto | Score |
|---|---|---|---|---|
| ¿Quién fue Tesla? | "físico del siglo XIX, corriente alterna" | ingeniero e inventor, AC, estudió física en 1870s | entailed=True | **1.0** |
| ¿Qué día es hoy? | "Lunes 13 de agosto 2026" | "11 de agosto 2026" | entailed=False | **0.0** |

> El contexto dice que Tesla *estudió física* y fue *ingeniero*, pero nunca dice "físico". El juez lo dio por soportado igual — un juez más estricto habría dicho `False`. Es un ejemplo real de inconsistencia del juez; lo retomamos en el bloque de sesgos.

---

## Implementación propia: faithfulness estilo DeepEval

In [10]:
GENERATE_TRUTHS = """\
Extrae las verdades o hechos presentes en el contexto recuperado. Cada verdad debe ser un \
enunciado atómico y verificable tomado únicamente del contexto. Devuelve también un breve \
resumen en español.
Contexto: {context}"""

In [11]:
VERIFY_DEEPEVAL = """\
¿Las siguientes verdades contradicen la afirmación? Asigna 0 SOLO si las verdades contradicen \
directamente la afirmación. Asigna 1 si la afirmación concuerda con las verdades o si no se \
menciona (no verificable). Justifica brevemente en español.
Verdades:
{truths}
Afirmación: {claim}"""

In [12]:
class Truths(BaseModel):
    """Ground-truth facts extracted from the retrieved context."""

    truths: list[str] = []
    summary: str = ""


class ScoreResponse(BaseModel):
    """The structured result a judge requests from the model for a rubric score."""

    score: float
    justification: str

In [13]:
def faithfulness_deepeval(prompt: str, response: str, context: str):

    truths = call_openai(
        messages=[
            {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": GENERATE_TRUTHS.format(**{"context": context}),
            },
        ],
        schema=Truths,
    )

    claims = split_sentences(response)

    verdicts = []
    for claim in claims:
        verdict = call_openai(
            messages=[
                {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": VERIFY_DEEPEVAL.format(
                        **{"truths": truths.truths, "claim": claim}  # type: ignore
                    ),
                },
            ],
            schema=ScoreResponse,
        )

        verdicts.append(verdict)

    print(verdicts)

    # score=1.0 para "yes" e "idk" (VERIFY_DEEPEVAL solo asigna 0 si hay contradicción directa)
    return sum([v.score for v in verdicts]) / len(verdicts)

In [14]:
faithfulness_deepeval(
    "Quien fue Nikola Tesla? y Cuáles fueron sus contribuciones?",
    "Fue un físico del siglo XIX y contribuyo al diseño de la corriente alterna",
    "Nikola Tesla fue un ingeniero, futurista e inventor serbio-estadounidense. Es conocido por sus contribuciones al diseño del sistema moderno de suministro eléctrico de corriente alterna. "
    "Nacido y criado en el Imperio austrohúngaro, Tesla estudió ingeniería y física en la década de 1870, aunque no obtuvo ningún título.",
)

[ScoreResponse(score=1.0, justification='La afirmación es correcta en indicar que Nikola Tesla fue un físico del siglo XIX y que contribuyó al diseño de la corriente alterna, lo cual es corroborado por las verdades proporcionadas. Ninguna de las verdades contradice directamente esta afirmación.')]


1.0

In [15]:
faithfulness_deepeval(
    "Que dia es hoy?",
    "Hoy es Lunes 13 de Agosto de 2026",
    "La fecha de hoy es 11 de Agosto de 2026",
)

[ScoreResponse(score=0.0, justification='Las verdades proporcionadas indican que la fecha es 11 de Agosto de 2026, lo que contradice directamente la afirmación de que es 13 de Agosto de 2026. Por lo tanto, se asigna un 0.')]


0.0

---

## Resultados DeepEval

| Pregunta | Respuesta | Contexto (resumen) | Score | Justificación |
|---|---|---|---|---|
| ¿Quién fue Tesla? | "físico del siglo XIX, corriente alterna" | ingeniero e inventor, AC, estudió física en 1870s | **1.0** | "es un hecho conocido" |
| ¿Qué día es hoy? | "Lunes 13 de agosto 2026" | "11 de agosto 2026" | **0.0** | contradice directamente |

---

## El mismo cálculo con la librería: `FaithfulnessMetric` de DeepEval

In [17]:
import os
import pathlib

from uuid import uuid4

from dotenv import load_dotenv

from typing import TypedDict

from pydantic import SecretStr, Field, BaseModel

from langchain.agents import create_agent
from langgraph.graph import StateGraph
from langchain.agents.middleware import dynamic_prompt
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import MarkdownHeaderTextSplitter

from deepeval.integrations.langchain import CallbackHandler
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.tracing import observe, update_current_span

load_dotenv()

True

---

## Vector Store

In [18]:
md_text_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "header1"),
        ("##", "header2"),
        ("###", "header3"),
    ],
    strip_headers=False,
)

path = pathlib.Path("../data/docs_rag")

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    check_embedding_ctx_length=False,
    api_key=SecretStr(os.environ["OPENAI_API_KEY"]),
)

vector_store = Chroma(
    collection_name="qbr",
    embedding_function=embeddings,
    persist_directory="../.chroma_langchain_db",
)

for file in path.glob("*.md"):
    with open(file) as blob:
        doc = blob.read()
    documents = md_text_splitter.split_text(doc)

    vector_store.add_documents(
        documents=documents,
        ids=[str(uuid4()) for _ in range(len(documents))],
    )

retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.2, "k": 50},
)

---

## Retrieval Evaluator

In [19]:
class RetrievalEvaluator(BaseModel):
    """Classify retrieved documents based on how relevant it is to the user's question."""

    binary_evaluation: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )


retrieval_evaluator = create_agent(
    model="gpt-5.4-nano-2026-03-17",
    response_format=RetrievalEvaluator,
    system_prompt=(
        "Eres un evaluador de recuperación de documentos responsable de determinar si un documento recuperado es relevante para la pregunta del usuario. "
        "Considera que un documento es relevante si contiene palabras clave, conceptos, información relacionada o significado semántico que contribuya a responder la pregunta del usuario. "
        "Emite únicamente una puntuación binaria: "
        '"yes": si el documento es relevante para la pregunta.'
        '"no": si el documento no es relevante para la pregunta.'
        'No agregues explicaciones ni información adicional. La salida debe ser exclusivamente "yes" o "no".'
    ),
)

---
## Question Rewritter

In [20]:
class Question(BaseModel):
    """Question rewritten to give more context to the RAG or web Search"""

    question: str = Field(
        description="question rewritten to give more context to the LLM"
    )


question_rewriter = create_agent(
    model="gpt-5.4-nano-2026-03-17",
    response_format=Question,
    system_prompt=(
        "Eres un reescritor de preguntas encargado de transformar una pregunta de entrada en una versión mejorada y optimizada para búsquedas mediante sistemas RAG (Retrieval-Augmented Generation). "
        "Analiza la pregunta de entrada e identifica su intención semántica subyacente, es decir, qué información busca realmente obtener el usuario. "
        "Devuelve únicamente la pregunta reescrita, sin explicaciones ni comentarios adicionales. "
    ),
)

---
## Main Agent

In [21]:
@dynamic_prompt
def build_prompt(request) -> str:
    documents = request.runtime.context["documents"]
    context_text = "\n\n".join(d.page_content for d in documents)
    return (
        "Eres un asistente especializado en responder preguntas utilizando información recuperada de un sistema RAG. "
        "Utiliza exclusivamente el contexto proporcionado para responder la pregunta del usuario. Si la información necesaria para responder no está disponible o no puedes determinar la respuesta con suficiente certeza a partir del contexto, indica que no lo sabes. "
        "Responde de forma clara y concisa, con un máximo de tres oraciones. "
        f"Contexto recuperado: {context_text}"
    )


main_agent = create_agent(
    model="gpt-5.4-nano-2026-03-17",
    middleware=[build_prompt],
)

---
## LangGraph

In [22]:
def regenerate_query(state):
    question = state["question"]

    better_question = question_rewriter.invoke(
        {"messages": [{"role": "user", "content": question}]}
    )

    return {
        "question": better_question["structured_response"].question,
    }


# Aquí entra el juez: FaithfulnessMetric/AnswerRelevancyMetric corren sobre el
# LLMTestCase que arma update_current_span más abajo.
@observe(
    metrics=[
        AnswerRelevancyMetric(verbose_mode=True),
        FaithfulnessMetric(verbose_mode=True),
    ]
)
def answer_question(state):
    question = state["question"]
    documents = retriever.invoke(question)

    answer = main_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        context={"documents": documents},  # type: ignore
    )

    update_current_span(
        test_case=LLMTestCase(
            input=question,
            actual_output=answer["messages"][-1].content,
            # aquí entra retrieval_context: sin esto, FaithfulnessMetric no tiene contra qué verificar
            retrieval_context=[d.page_content for d in documents],
        )
    )

    return {
        "question": question,
        "answer": answer["messages"][-1].content,
    }

In [23]:
class RagState(TypedDict):
    question: str
    answer: str


workflow = StateGraph(RagState)
workflow.add_node("regenerate_query_node", regenerate_query)
workflow.add_node("answer_question_node", answer_question)

workflow.add_edge("__start__", "regenerate_query_node")
workflow.add_edge("regenerate_query_node", "answer_question_node")
workflow.add_edge("answer_question_node", "__end__")


app = workflow.compile()

In [24]:
dataset = EvaluationDataset(
    goldens=[
        Golden(
            input="Que problema se concentra en los tickets de Initech?",
            multimodal=False,
        ),
        Golden(
            input="Porque la cuenta Acme declino en el Q2 de este año?",
            multimodal=False,
        ),
    ]
)

for golden in dataset.evals_iterator():
    app.invoke(
        {
            "question": golden.input,
            "answer": "",
        },
        config={"callbacks": [CallbackHandler()]},
    )

Output()

**************************************************

Answer Relevancy Verbose Logs

**************************************************

Statements:
[
    "En Q2-2026, Acme mostró señales de deterioro.",
    "Las caídas del servicio en producción en abril y mayo dispararon múltiples incidentes P1.",
    "Las caídas del servicio generaron reportes post-incidente formales.",
    "Las caídas del servicio terminaron en notas de crédito por incumplimiento de SLA.",
    "Esto explica la caída de ingresos reconocidos del trimestre.",
    "La cuenta puede aparentar estar sana por ingresos planos.",
    "Se observa churn silencioso.",
    "El uso se desplomó.",
    "Los usuarios activos están a la baja.",
    "Los logins están a la baja.",
    "La adopción está a la baja.",
    "Casi no abren tickets.",
    "Eso es una señal de desenganche.",
    "La salida del champion (Director de TI) ocurrió en mayo.",
    "Hay una evaluación activa de un competidor, StratusOne.",
    "StratusOne ofrece migración asistida y descuento agresivo.",
    "Esto puede estar acelerando la declinación.",
    "La presión aumenta especialmente hacia la renovación de Q3-2026."
] 
 
Verdicts:
[
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "no",
        "reason": "El identificador del ticket no ayuda a determinar el problema espec\u00edfico o la 
categor\u00eda principal de incidencias m\u00e1s frecuente en los tickets de Initech."
    }
]
 
Score: 0.6666666666666666
Reason: The score is 0.67 because the answer appears to address the main question about the most frequent 
issue/category in Initech tickets, but it is not higher because it included an irrelevant statement about the 
ticket identifier not helping determine the specific problem, which does not contribute to identifying the central 
issue.

...

Truths (limit=None):
[
    "Existe un ticket de soporte de Initech con el identificador TCK-00079 titulado "¿Cómo asigno roles a un nuevo 
usuario?".",
    "El ticket TCK-00079 tiene fecha 2026-01-08.",
    "El ticket TCK-00079 está categorizado como "Consulta / Cómo hacer".",
    "El ticket TCK-00079 tiene prioridad P3.",
    "En el ticket TCK-00079, Initech pregunta cómo asignar permisos por rol en Nimbus Core y solicita una guía paso
a paso.",
    "El texto incluye un contexto del trimestre Q2-2026.",
    "En el contexto de Q2-2026, los ingresos se describen como planos.",
    "En el contexto de Q2-2026, se menciona una fricción creciente alrededor del módulo de reportes y tableros.",
    "En el contexto de Q2-2026, aumentaron los tickets de bugs relacionados con filtros que no se guardan y 
exportaciones vacías.",
    "En el contexto de Q2-2026, aumentaron las solicitudes de reportes personalizados.",
    "En el contexto de Q2-2026, la CSAT bajó ligeramente.",
    "Existe un ticket de soporte de Acme Corporation con el identificador TCK-00002 titulado "Configuración de 
notificaciones".",
    "El ticket TCK-00002 tiene fecha 2026-01-01.",
    "El ticket TCK-00002 está categorizado como "Consulta / Cómo hacer".",
    "El ticket TCK-00002 tiene prioridad P3.",
    "En el ticket TCK-00002, Acme Corporation quiere ajustar las notificaciones por correo de Nimbus Core y 
pregunta dónde se configura el umbral de alertas.",
    "Existe un ticket de soporte de Globex S.A. con el identificador TCK-00048 titulado "Necesitamos exportar 
reportes a Excel".",
    "El ticket TCK-00048 tiene fecha 2026-01-09.",
    "El ticket TCK-00048 está categorizado como "Solicitud de función".",
    "El ticket TCK-00048 tiene prioridad P4.",
    "En el ticket TCK-00048, Globex S.A. solicita exportar los tableros de Nimbus Core a Excel y PDF de forma 
nativa.",
    "En el ticket TCK-00048, Globex S.A. indica que actualmente realiza esas exportaciones manualmente y que eso 
consume mucho tiempo.",
    "En el ticket TCK-00048, Globex S.A. pregunta si esa capacidad está en el roadmap.",
    "Existe un ticket con el identificador TCK-00012 titulado "Intermitencias y errores 500".",
    "El ticket TCK-00012 tiene fecha 2026-04-15.",
    "El ticket TCK-00012 está categorizado como "Caída del servicio".",
    "El ticket TCK-00012 tiene prioridad P2.",
    "En el ticket TCK-00012, se reporta que Nimbus Core responde de forma intermitente con errores 500 desde esa 
mañana.",
    "En el ticket TCK-00012, se afirma que es la segunda caída de ese mes.",
    "En el ticket TCK-00012, se indica que el equipo de operaciones está muy molesto.",
    "En el ticket TCK-00012, se pregunta si hay un incidente abierto.",
    "Existe un ticket con el identificador TCK-00023 titulado "Intermitencias y errores 500".",
    "El ticket TCK-00023 tiene fecha 2026-04-10.",
    "El ticket TCK-00023 está categorizado como "Caída del servicio".",
    "El ticket TCK-00023 tiene prioridad P2.",
    "En el ticket TCK-00023, se reporta que Nimbus Core responde de forma intermitente con errores 500 desde esa 
mañana.",
    "En el ticket TCK-00023, se afirma que es la tercera caída de ese mes.",
    "En el ticket TCK-00023, se indica que el equipo de operaciones está muy molesto.",
    "En el ticket TCK-00023, se pregunta si hay un incidente abierto.",
    "Existe un ticket con el identificador TCK-00001 titulado "Duda sobre el cobro de asientos adicionales".",
    "El ticket TCK-00001 tiene fecha 2026-01-20.",
    "El ticket TCK-00001 está categorizado como "Facturación".",
    "El ticket TCK-00001 tiene prioridad P3.",
    "En el ticket TCK-00001, se reporta un incremento en el monto facturado.",
    "En el ticket TCK-00001, se solicita el desglose de asientos y el prorrateo aplicado ese mes.",
    "Existe un ticket con el identificador TCK-00004 titulado "Duda sobre el cobro de asientos adicionales".",
    "El ticket TCK-00004 tiene fecha 2026-01-07.",
    "El ticket TCK-00004 está categ

======================================================================

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 0 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 0 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ answer_question                                                                                             │
│  ├──   Input:            ¿Por qué la cuenta “Acme” registró una disminución (declinó) en el Q2 de este año?     │
│  │                       Detalla las causas probables y, si aplica, referencia factores como desempeño          │
│  │                       financiero, ingresos, costos, variaciones interanuales/trimestrales, eventos           │
│  │                       internos o cambios de mercado.                                                         │
│  │     Actual Output:    En Q2-2026, “Acme” mostró señales de deterioro principalmente por **caídas del         │
│  │                       servicio en producción en abril y mayo** que dispararon **múltiples incidentes         │
│  │                       P1**, generaron **reportes post-incidente formales** y terminaron en **notas de        │
│  │                       crédito por incumplimiento de SLA**, lo que explica la **caída de ingresos             │
│  │                       reconocidos** del trimestre. Además, pese a que la cuenta puede aparentar “estar       │
│  │                       sana” por **ingresos planos**, se observa **churn silencioso**: el **uso se            │
│  │                       desplomó** (usuarios activos, logins y adopción a la baja) y casi no abren             │
│  │                       tickets—señal de desenganche. Finalmente, hay factores comerciales/organizacionales    │
│  │                       que aumentan la presión: **salida del champion (Director de TI) en mayo** y            │
│  │                       **evaluación activa de un competidor (StratusOne) con migración asistida y             │
│  │                       descuento agresivo**, lo que puede estar acelerando la declinación, especialmente      │
│  │                       hacia la renovación de **Q3-2026**.                                                    │
│  └── Metrics                                                                                                    │
│       Status  ┃ Metric            ┃ Score  ┃ Threshold  ┃ Reason                                                │
│      ━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS   │ Answer Relevancy  │ 1.00   │ 0.50       │ The score is 1.00 because the response appears ...    │
│        PASS   │ Faithfulness      │ 1.00   │ 0.50       │ The score is 1.00 because there are no contradi...    │
│                                                                                                                 │
╰───────────────────────────────────────────────────────────

⚠ WARNING: No hyperparameters logged.
» ]8;id=1914151;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 19.32s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 100.0% | Passed: 2 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

---

## Y así se ve la respuesta a la pregunta del título

| Golden | Faithfulness | Answer Relevancy | Razón principal del juez |
|---|---|---|---|
| ¿Qué problema se concentra en los tickets de Initech? | **1.00** | **1.00** | Faithfulness: "no hay contradicciones, la salida es consistente con el contexto recuperado." |
| ¿Por qué la cuenta Acme declinó en el Q2 de este año? | **1.00** | **1.00** | Answer Relevancy: "la respuesta es totalmente relevante, sin afirmaciones irrelevantes — buen foco en el análisis pedido sobre por qué declinó Acme en Q2." |

**Agregado (2 test cases, `verbose_mode=True`):** Faithfulness avg 1.00 (2/2 pass) · Answer Relevancy avg 1.00 (2/2 pass) · evaluación completa en 19.32s.

Con dos goldens ya tienes un número por pregunta y una razón por número.

> **Bonus no buscado:** el log verbose crudo mostró en un punto `Score: 0.67` para Answer Relevancy con una razón sobre el ticket de Initech, pero con la lista de `statements` de la respuesta de Acme — un artefacto de evaluar los dos goldens de forma concurrente con `evals_iterator` (el objeto de la métrica comparte estado interno entre corridas async). El score oficial que DeepEval registró en la tabla final para ambos casos fue 1.00. Es, sin buscarlo, un ejemplo real de por qué correr al juez de forma concurrente sin aislamiento puede producir salidas de debug engañosas.

---

## Trampa 1: el juez tiene sesgos, y son predecibles

| Sesgo | Qué hace el juez | Mitigación |
|---|---|---|
| Posición | Prefiere la propuesta que aparece primera o última en el prompt, según el modelo | Intercambiar el orden y promediar |
| Verbosidad | Premia respuestas largas y prolijas sobre una breve y clara | Normalizar por longitud |
| Auto-preferencia | Favorece redacciones de su propia familia de modelos | Ensemble de familias distintas |
| Sensibilidad al prompt | Un cambio en la rúbrica desajusta las puntuaciones finales | Versionar rúbricas + set de control |

---

## Trampa 2: el juez cambia debajo de ti

**Deriva del modelo**
Los proveedores actualizan sus modelos de forma constante y sin previo aviso; eso puede desajustar las calificaciones a lo largo del tiempo.
→ Mitigación: fijar la versión del juez y re-evaluar periódicamente un set de control.

**Optimización adversarial**
Si entrenas un modelo usando el mismo juez como señal de recompensa, puede aprender los patrones que maximizan la puntuación en lugar de dar mejores respuestas.
→ Mitigación: el juez de evaluación no debe ser el mismo juez usado para entrenar/optimizar.

---

## Calibración: hacer que el score signifique lo mismo hoy, mañana y en otro modelo

**Nivel prompt (juez vía API — siempre aplicable)**
- Intercambio de orden (mitiga sesgo de posición)
- Normalización de longitud (mitiga sesgo de verbosidad)
- Ensembles de familias de modelos distintas (mitiga auto-preferencia)
- Forzar razonamiento antes del veredicto

**Nivel modelo (requiere pesos abiertos — no aplica a un juez vía API)**
- Calibración por sondeo (linear probes): de las capas intermedias se extraen activaciones y se ajusta un clasificador que predice si el veredicto del juez será correcto o incorrecto — técnica y resultados en Radharapu et al., *Calibrating LLM Judges: Linear Probes for Fast and Reliable Uncertainty Estimation* (ver referencias).
- ¿Por qué capas intermedias? Las capas iniciales son de bajo nivel, con poca representación rica; las capas finales están muy especializadas en predecir el siguiente token; las capas intermedias retienen la mayor representación semántica de toda la red.

**Medir la calibración**
- Expected Calibration Error (ECE): desajuste entre la confianza predicha y la precisión real — es la métrica que usa el paper de Radharapu et al. para evaluar los linear probes.
- ~~Estadístico de Kuiper~~ — verificado: la fuente de sondeo citada arriba no lo usa ni lo menciona. Quitar de la charla o buscar una referencia distinta antes de presentar.

---

## ¿Cuánto cuesta evaluar a volumen real?

Faithfulness no es una llamada: es N+1 llamadas por respuesta (RAGAS) o 2N+1 (DeepEval).

- **RAGAS:** 1 extracción de statements + N verificaciones (una por afirmación)
- **DeepEval:** 1 extracción de truths + 1 extracción de claims + N veredictos
- **Variables:** número de afirmaciones N, tamaño del contexto, modelo juez elegido
- **Palancas:** juez más pequeño para veredictos binarios · cache de `truths` por contexto (se reutiliza entre preguntas con el mismo contexto) · muestreo estratificado en vez de evaluar el 100%

**Costo estimado por 1,000 respuestas evaluadas**, según tamaño de contexto por pregunta:

| Juez | 500 tok/pregunta | 1,000 tok/pregunta | Con Batch API (−50%) |
|---|---|---|---|
| Opus 5 | $7.50 | $10.00 | $3.75 – $5.00 |
| Sonnet 5 | $3.00 | $4.00 | $1.50 – $2.00 |
| Sol (lista $5/$30) | $8.50 | $11.00 | $4.25 – $5.50 |
| Sol (promo $4/$20) | $6.00 | $8.00 | $3.00 – $4.00 |
| Terra | $3.40 | $4.40 | $1.70 – $2.20 |

Un juez más barato (Sonnet 5, Terra) puede costar 2–3x menos que uno de gama alta (Opus 5, Sol) para la misma cobertura — y la Batch API corta el costo a la mitad en cualquiera de los dos, al precio de perder evaluación en tiempo real.

---

## Lo que te llevas

1. Faithfulness mide fidelidad al contexto, no verdad.
2. RAGAS exige soporte positivo; DeepEval solo castiga contradicción. Elige sabiendo cuál quieres.
3. Un juez sin calibración es una opinión con formato JSON.

*Contextual precision y contextual recall quedan para la siguiente charla.*

Código: github.com/jmanuelc87

---

# ¡Muchas gracias!

**Referencias**
- GitHub: github.com/jmanuelc87
- LinkedIn: linkedin.com/in/jmanuelc87
- RAGAS — Faithfulness: https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/faithfulness.html
- DeepEval — FaithfulnessMetric: https://deepeval.com/docs/metrics-faithfulness
- Radharapu, Saxena, Li, Whitehouse, Williams, Cancedda — *Calibrating LLM Judges: Linear Probes for Fast and Reliable Uncertainty Estimation*: https://arxiv.org/abs/2512.22245